# ARR Forecast — Prophet Model on Databricks

Runs Meta's Prophet time series model **directly on the Databricks cluster** where Won ARR data lives.
Matches the methodology from the ARR Forecast Power BI dashboard (PPT reference):
- **Data source**: `federated.sales.metis_won_opps_fact` (Won ARR by Geo × Product)
- **Segmentation**: NA / EMEA / LATAM / APAC × ITSG / UCC + Total
- **Scenarios**: Worst Case (`yhat_lower`) · Most Likely (`yhat`) · Best Case (`yhat_upper`)
- **Confidence interval**: 80% (matches Power BI dashboard)
- **Output**: Results written to `datagroup_mdl.mdl_sales_analytics.arr_forecast_prophet`
- **App connection**: FastAPI backend reads from the Delta output table via SQL

**Run schedule**: Daily via Databricks Workflow (or on-demand before dashboard refresh)

## Section 1 — Install and Import Required Libraries

In [ ]:
# Install Prophet and plotting support on the cluster
# Run once per cluster; subsequent runs use the cached install
%pip install prophet==1.1.5 --quiet

# Standard imports
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import matplotlib
matplotlib.use("Agg")          # non-interactive backend for cluster notebooks
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# Prophet
from prophet import Prophet

# PySpark (already available on Databricks)
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, DateType, TimestampType, BooleanType
)

# MLflow for model registry
import mlflow
import mlflow.prophet

spark = SparkSession.builder.getOrCreate()
print(f"Spark version: {spark.version}")
print(f"Prophet imported OK")
print(f"MLflow version: {mlflow.__version__}")

## Section 2 — Configuration

Edit these parameters to control data source, output table, and forecast horizon.

In [ ]:
# ── Data source ──────────────────────────────────────────────────────────────
SOURCE_TABLE   = "federated.sales.metis_won_opps_fact"
HISTORY_START  = "2022-01-01"   # use data from this date forward for training

# ── Output ───────────────────────────────────────────────────────────────────
OUTPUT_CATALOG = "datagroup_mdl"
OUTPUT_SCHEMA  = "mdl_sales_analytics"
OUTPUT_TABLE   = f"{OUTPUT_CATALOG}.{OUTPUT_SCHEMA}.arr_forecast_prophet"

# ── Forecast horizon ─────────────────────────────────────────────────────────
FORECAST_WEEKS = 52          # how many weeks ahead to forecast (1 year)
FREQ           = "W-MON"     # weekly, week starts Monday (ISO standard)

# ── Prophet hyperparameters (matching ARR Forecast Power BI config) ───────────
INTERVAL_WIDTH         = 0.80   # 80% CI → Best Case / Worst Case bands
CHANGEPOINT_PRIOR      = 0.05   # flexibility of trend; lower = smoother
SEASONALITY_MODE       = "multiplicative"   # multiplicative fits ARR growth better

# ── Segmentation dimensions ──────────────────────────────────────────────────
GEO_VALUES     = ["APAC", "EMEA", "LATAM", "NA", "UNKNOWN", "Total"]
PRODUCT_VALUES = ["ITSG", "UCC", "Total"]

RUN_DATE = datetime.now().strftime("%Y-%m-%d")
print(f"Config loaded. Run date: {RUN_DATE}")
print(f"Output table: {OUTPUT_TABLE}")
print(f"Forecast: {FORECAST_WEEKS} weeks ahead from {RUN_DATE}")

## Section 3 — Connect to Databricks and Load Won ARR Data

Reads `federated.sales.metis_won_opps_fact` — the latest snapshot of all closed-won opportunities.
Uses `data_date = MAX(data_date)` to get the most current snapshot without duplicates.

In [ ]:
# Load Won ARR from the federated Metis table
# Uses the latest data_date snapshot to avoid double-counting
sql = f"""
SELECT
    close_date,
    COALESCE(UPPER(TRIM(sales_market)), 'UNKNOWN')  AS geo,
    COALESCE(UPPER(TRIM(product_group)), 'UNKNOWN') AS product_group,
    SUM(amount_towards_plan)                         AS won_arr
FROM {SOURCE_TABLE}
WHERE
    data_date = (SELECT MAX(data_date) FROM {SOURCE_TABLE})
    AND close_date >= '{HISTORY_START}'
    AND close_date <= CURRENT_DATE()
    AND amount_towards_plan IS NOT NULL
    AND amount_towards_plan > 0
GROUP BY
    close_date,
    COALESCE(UPPER(TRIM(sales_market)), 'UNKNOWN'),
    COALESCE(UPPER(TRIM(product_group)), 'UNKNOWN')
ORDER BY close_date
"""

sdf_raw = spark.sql(sql)
print(f"Rows loaded: {sdf_raw.count():,}")
print(f"Date range:")
sdf_raw.selectExpr("MIN(close_date) AS min_date", "MAX(close_date) AS max_date").show()
print("Geo values:")
sdf_raw.groupBy("geo").agg(F.sum("won_arr").alias("total_arr")).orderBy("geo").show()
print("Product values:")
sdf_raw.groupBy("product_group").agg(F.sum("won_arr").alias("total_arr")).orderBy("product_group").show()

## Section 4 — Prepare Data for Prophet

Aggregates daily close_date data to **weekly** (ISO Monday-aligned) buckets.
Adds the seasonality feature columns used by the Power BI model: `summer`, `winter`, `isweek1`.

In [ ]:
def prepare_prophet_df(pdf: pd.DataFrame) -> pd.DataFrame:
    """
    Convert a filtered pandas DataFrame to Prophet format.
    - Renames close_date → ds, won_arr → y
    - Aggregates to ISO-week (Monday-start)
    - Adds seasonality regressors matching ARR Forecast Power BI model:
        summer   = June / July / August
        winter   = December / January / February
        isweek1  = first 7 days of the month
    - Fills missing weeks with 0 (no deals closed that week)
    """
    df = pdf.copy()
    df["ds"] = pd.to_datetime(df["close_date"])
    df["y"]  = df["won_arr"].astype(float)

    # Aggregate to weekly buckets (ISO Monday start)
    df["week_start"] = df["ds"].dt.to_period(FREQ).apply(lambda p: p.start_time)
    weekly = (
        df.groupby("week_start", as_index=False)["y"]
        .sum()
        .rename(columns={"week_start": "ds"})
    )

    # Fill the full calendar range so Prophet sees a complete weekly series
    full_range = pd.date_range(
        start=weekly["ds"].min(),
        end=weekly["ds"].max(),
        freq=FREQ,
    )
    weekly = weekly.set_index("ds").reindex(full_range, fill_value=0).reset_index()
    weekly.columns = ["ds", "y"]
    weekly["ds"] = pd.to_datetime(weekly["ds"])

    # Seasonality regressors (matching PPT feature engineering)
    weekly["summer"]  = weekly["ds"].dt.month.isin([6, 7, 8]).astype(int)
    weekly["winter"]  = weekly["ds"].dt.month.isin([12, 1, 2]).astype(int)
    weekly["isweek1"] = (weekly["ds"].dt.day <= 7).astype(int)

    return weekly.sort_values("ds").reset_index(drop=True)


# Convert to pandas and build a "Total" aggregate in addition to segments
df_raw = sdf_raw.toPandas()
df_raw["close_date"] = pd.to_datetime(df_raw["close_date"])

# Build total series
df_total = df_raw.groupby("close_date", as_index=False)["won_arr"].sum()
df_total["geo"] = "Total"
df_total["product_group"] = "Total"

df_all = pd.concat([df_raw, df_total], ignore_index=True)

# Preview the Total series prepared for Prophet
sample = prepare_prophet_df(df_total)
print(f"Total series: {len(sample)} weekly points  ({sample['ds'].min().date()} → {sample['ds'].max().date()})")
print(f"Average weekly Won ARR (Total): ${sample['y'].mean():,.0f}")
print(f"\nFirst 5 rows:")
print(sample.head())

## Section 5 — Build and Train Prophet Model

Instantiates one Prophet model per segment (Geo × Product combination).
Configuration exactly matches the ARR Forecast Power BI dashboard methodology.

In [ ]:
def build_and_train_prophet(weekly_df: pd.DataFrame, segment_label: str) -> tuple:
    """
    Build and fit a Prophet model for one Geo×Product segment.

    Returns: (fitted_model, forecast_df, mape_pct)
    """
    if len(weekly_df) < 26:
        print(f"  [SKIP] {segment_label}: only {len(weekly_df)} weeks — need at least 26")
        return None, None, None

    # ── Prophet config matching ARR Forecast Power BI ─────────────────────────
    model = Prophet(
        interval_width         = INTERVAL_WIDTH,       # 0.80 → Best/Worst case bands
        changepoint_prior_scale= CHANGEPOINT_PRIOR,    # 0.05 → smooth trend
        seasonality_mode       = SEASONALITY_MODE,     # multiplicative
        yearly_seasonality     = True,
        weekly_seasonality     = True,
        daily_seasonality      = False,                # weekly data — no daily pattern
    )

    # Custom seasonality regressors (PPT Feature Engineering step)
    model.add_regressor("summer")
    model.add_regressor("winter")
    model.add_regressor("isweek1")

    # Fit on the full history
    model.fit(weekly_df[["ds", "y", "summer", "winter", "isweek1"]])

    # ── Accuracy on held-out last 8 weeks ─────────────────────────────────────
    train_df  = weekly_df.iloc[:-8]
    test_df   = weekly_df.iloc[-8:]
    eval_model = Prophet(
        interval_width=INTERVAL_WIDTH,
        changepoint_prior_scale=CHANGEPOINT_PRIOR,
        seasonality_mode=SEASONALITY_MODE,
        yearly_seasonality=True,
        weekly_seasonality=True,
        daily_seasonality=False,
    )
    eval_model.add_regressor("summer")
    eval_model.add_regressor("winter")
    eval_model.add_regressor("isweek1")
    eval_model.fit(train_df[["ds", "y", "summer", "winter", "isweek1"]])

    test_pred = eval_model.predict(test_df[["ds", "summer", "winter", "isweek1"]])
    actuals   = test_df["y"].values
    preds     = test_pred["yhat"].values
    mape = float(np.mean(np.abs((actuals - preds) / np.where(actuals == 0, 1, actuals)))) * 100

    # ── Generate forecast ─────────────────────────────────────────────────────
    future = model.make_future_dataframe(periods=FORECAST_WEEKS, freq=FREQ)
    future["summer"]  = future["ds"].dt.month.isin([6, 7, 8]).astype(int)
    future["winter"]  = future["ds"].dt.month.isin([12, 1, 2]).astype(int)
    future["isweek1"] = (future["ds"].dt.day <= 7).astype(int)

    forecast_df = model.predict(future)

    print(f"  [OK] {segment_label:<22}  weeks={len(weekly_df)}  MAPE={mape:.1f}%")
    return model, forecast_df, mape


# ── Train all segments ─────────────────────────────────────────────────────────
all_results = {}   # key: (geo, product_group)  → (model, forecast_df, mape)

segments = df_all.groupby(["geo", "product_group"])

print(f"Training {segments.ngroups} segments...\n")

for (geo, product), group_df in segments:
    label = f"{geo}/{product}"
    weekly = prepare_prophet_df(group_df)
    model, fc_df, mape = build_and_train_prophet(weekly, label)
    if model is not None:
        all_results[(geo, product)] = {"model": model, "forecast": fc_df, "mape": mape, "history": weekly}

print(f"\n✓ Trained {len(all_results)} segments successfully")

## Section 6 — Generate & Review Forecast Results

Assembles all segment forecasts into a single flat DataFrame with columns matching the Power BI dashboard output format.

In [ ]:
rows = []
history_cutoff = pd.Timestamp(RUN_DATE)

for (geo, product), result in all_results.items():
    fc_df   = result["forecast"]
    history = result["history"]
    mape    = result["mape"]

    # Build a lookup for actual values (historical weeks)
    actual_map = dict(zip(history["ds"], history["y"]))

    for _, row in fc_df.iterrows():
        week_date     = row["ds"]
        is_historical = bool(week_date <= history_cutoff)
        actual_val    = actual_map.get(week_date, None)

        rows.append({
            "run_date"      : RUN_DATE,
            "geo"           : geo,
            "product_group" : product,
            "forecast_date" : week_date.date(),
            "most_likely"   : max(0.0, round(float(row["yhat"]),       2)),
            "worst_case"    : max(0.0, round(float(row["yhat_lower"]), 2)),
            "best_case"     : max(0.0, round(float(row["yhat_upper"]), 2)),
            "actual"        : round(float(actual_val), 2) if actual_val is not None else None,
            "is_historical" : is_historical,
            "model_mape"    : round(mape, 2),
        })

df_output = pd.DataFrame(rows)
print(f"Total output rows: {len(df_output):,}")
print(f"  Historical rows : {df_output['is_historical'].sum():,}")
print(f"  Forecast rows   : {(~df_output['is_historical']).sum():,}")

In [ ]:
# Fix the preview — simpler display without the pipe operator
sample_view = (
    df_output
    .query("geo == 'Total' and product_group == 'Total' and is_historical == False")
    [["forecast_date", "worst_case", "most_likely", "best_case"]]
    .head(8)
)
print("Total segment — next 8 forecast weeks:")
print(sample_view.to_string(index=False))

## Section 7 — Visualize Forecast Results

Plots the Total segment: actual history + 3 scenario bands (Worst / Most Likely / Best Case).
Matches the "Actual Won ARR vs Predicted" chart from the Power BI dashboard.

In [ ]:
def plot_forecast(geo: str, product: str, lookback_weeks: int = 52):
    """
    Plot actuals + 3 scenario bands for a given Geo × Product segment.
    Matches the 'Actual Won ARR vs Predicted' Power BI chart style.
    """
    df_seg = df_output.query(f"geo == '{geo}' and product_group == '{product}'").copy()
    df_seg["forecast_date"] = pd.to_datetime(df_seg["forecast_date"])
    df_seg = df_seg.sort_values("forecast_date")

    hist = df_seg[df_seg["is_historical"]]
    fcast = df_seg[~df_seg["is_historical"]]

    # Show only recent history for readability
    cutoff = df_seg["forecast_date"].max() - pd.Timedelta(weeks=lookback_weeks)
    hist = hist[hist["forecast_date"] >= cutoff]

    fig, axes = plt.subplots(2, 1, figsize=(14, 9))
    fig.suptitle(f"Won ARR Forecast — {geo} / {product}", fontsize=14, fontweight="bold")

    # ── Top: Weekly view ──────────────────────────────────────────────────────
    ax = axes[0]
    ax.plot(hist["forecast_date"], hist["actual"], color="#FFA500", linewidth=1.8,
            label="Actuals", zorder=5)
    ax.plot(fcast["forecast_date"], fcast["most_likely"], color="#FFA500",
            linewidth=1.5, linestyle="--", label="Most Likely")
    ax.fill_between(fcast["forecast_date"],
                    fcast["worst_case"], fcast["best_case"],
                    alpha=0.25, color="#888888", label="Worst–Best band")
    ax.plot(fcast["forecast_date"], fcast["worst_case"],
            color="white", linewidth=0.8, linestyle=":")
    ax.plot(fcast["forecast_date"], fcast["best_case"],
            color="white", linewidth=0.8, linestyle=":")
    ax.axvline(pd.Timestamp(RUN_DATE), color="red", linewidth=0.8, linestyle="--", alpha=0.6)
    ax.set_title("Weekly Forecast vs Actuals", fontsize=10)
    ax.set_ylabel("Won ARR ($)")
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"${x/1e6:.1f}M"))
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.set_facecolor("#1a1a1a")
    fig.patch.set_facecolor("#1a1a1a")
    ax.tick_params(colors="white")
    ax.yaxis.label.set_color("white")
    ax.title.set_color("white")

    # ── Bottom: Running totals ────────────────────────────────────────────────
    ax2 = axes[1]
    df_all_seg = pd.concat([
        hist[["forecast_date", "actual", "most_likely", "worst_case", "best_case"]],
        fcast[["forecast_date", "actual", "most_likely", "worst_case", "best_case"]],
    ]).sort_values("forecast_date")

    df_all_seg["cum_actual"]      = df_all_seg["actual"].fillna(0).cumsum()
    df_all_seg["cum_most_likely"] = df_all_seg["most_likely"].cumsum()
    df_all_seg["cum_worst"]       = df_all_seg["worst_case"].cumsum()
    df_all_seg["cum_best"]        = df_all_seg["best_case"].cumsum()

    ax2.plot(df_all_seg["forecast_date"], df_all_seg["cum_actual"],
             color="#FFA500", linewidth=1.8, label="Running Actuals")
    ax2.plot(df_all_seg["forecast_date"], df_all_seg["cum_most_likely"],
             color="#FFA500", linewidth=1.2, linestyle="--", label="Running Most Likely")
    ax2.fill_between(df_all_seg["forecast_date"],
                     df_all_seg["cum_worst"], df_all_seg["cum_best"],
                     alpha=0.2, color="#888888")
    ax2.axvline(pd.Timestamp(RUN_DATE), color="red", linewidth=0.8, linestyle="--", alpha=0.6)
    ax2.set_title("Running Totals", fontsize=10)
    ax2.set_ylabel("Cumulative Won ARR ($)")
    ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"${x/1e6:.0f}M"))
    ax2.legend(fontsize=8)
    ax2.grid(True, alpha=0.3)
    ax2.set_facecolor("#1a1a1a")
    ax2.tick_params(colors="white")
    ax2.yaxis.label.set_color("white")
    ax2.title.set_color("white")

    plt.tight_layout()
    plt.show()
    mape_val = df_output.query(
        f"geo == '{geo}' and product_group == '{product}'"
    )["model_mape"].iloc[0]
    print(f"Model MAPE (8-week holdout): {mape_val:.1f}%")


# Plot Total and key segments
plot_forecast("Total", "Total")
plot_forecast("NA",   "Total")
plot_forecast("EMEA", "Total")

## Section 8 — Save Results to Delta Table

Writes the full forecast output to `datagroup_mdl.mdl_sales_analytics.arr_forecast_prophet`.
The FastAPI app reads from this table — no model running in the app at query time.

In [ ]:
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, DateType, BooleanType
)

# Define explicit schema so Delta gets the right types
schema = StructType([
    StructField("run_date",       StringType(),  False),
    StructField("geo",            StringType(),  False),
    StructField("product_group",  StringType(),  False),
    StructField("forecast_date",  StringType(),  False),   # written as string, cast in SQL
    StructField("most_likely",    DoubleType(),  True),
    StructField("worst_case",     DoubleType(),  True),
    StructField("best_case",      DoubleType(),  True),
    StructField("actual",         DoubleType(),  True),
    StructField("is_historical",  BooleanType(), False),
    StructField("model_mape",     DoubleType(),  True),
])

# Convert dates to strings for Spark compatibility
df_to_write = df_output.copy()
df_to_write["forecast_date"] = df_to_write["forecast_date"].astype(str)
df_to_write["actual"] = df_to_write["actual"].where(df_to_write["actual"].notna(), other=None)

sdf_output = spark.createDataFrame(df_to_write, schema=schema)

# Create schema if not exists
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {OUTPUT_CATALOG}.{OUTPUT_SCHEMA}")

# Write as Delta, replacing today's run (INSERT OVERWRITE partition)
(
    sdf_output
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .option("replaceWhere", f"run_date = '{RUN_DATE}'")
    .saveAsTable(OUTPUT_TABLE)
)

row_count = spark.table(OUTPUT_TABLE).filter(F.col("run_date") == RUN_DATE).count()
print(f"✓ Written {row_count:,} rows to {OUTPUT_TABLE} (run_date = {RUN_DATE})")

# Quick validation
spark.sql(f"""
    SELECT geo, product_group, COUNT(*) AS rows,
           ROUND(AVG(model_mape), 1) AS avg_mape_pct
    FROM {OUTPUT_TABLE}
    WHERE run_date = '{RUN_DATE}'
    GROUP BY geo, product_group
    ORDER BY geo, product_group
""").show(30)

## Section 9 — Register Model to MLflow

Logs the Total segment Prophet model to the Databricks MLflow Model Registry.
The FastAPI app (or a Databricks App) can then call the model endpoint via REST instead of querying the Delta table.

In [ ]:
MLFLOW_EXPERIMENT = "/Users/gaim-team/arr-forecast-prophet"
REGISTERED_MODEL  = "arr_forecast_prophet_total"

mlflow.set_experiment(MLFLOW_EXPERIMENT)

total_result = all_results.get(("Total", "Total"))
if total_result is None:
    print("Total segment model not found — skipping MLflow registration")
else:
    total_model = total_result["model"]
    total_mape  = total_result["mape"]
    total_weeks = len(total_result["history"])

    with mlflow.start_run(run_name=f"arr_forecast_prophet_{RUN_DATE}") as run:
        # Log parameters matching Power BI config
        mlflow.log_params({
            "forecast_weeks"        : FORECAST_WEEKS,
            "interval_width"        : INTERVAL_WIDTH,
            "changepoint_prior_scale": CHANGEPOINT_PRIOR,
            "seasonality_mode"      : SEASONALITY_MODE,
            "frequency"             : FREQ,
            "training_weeks"        : total_weeks,
        })

        # Log accuracy metrics
        mlflow.log_metrics({
            "total_mape_pct" : round(total_mape, 2),
            "segments_trained": len(all_results),
        })

        # Log the Prophet model itself
        mlflow.prophet.log_model(
            pr_model=total_model,
            artifact_path="prophet_model",
            registered_model_name=REGISTERED_MODEL,
        )

        run_id = run.info.run_id
        print(f"✓ MLflow run logged:  {run_id}")
        print(f"  Experiment : {MLFLOW_EXPERIMENT}")
        print(f"  Model      : {REGISTERED_MODEL}")
        print(f"  Total MAPE : {total_mape:.1f}%")
        print(f"  Run date   : {RUN_DATE}")

## Section 10 — Connect to the FastAPI / Databricks App

### Architecture

```
Databricks Cluster (this notebook)
  └─ runs Prophet per-segment
  └─ writes → datagroup_mdl.mdl_sales_analytics.arr_forecast_prophet (Delta)

FastAPI backend (atlas-executive-insights)
  └─ GET /api/arr/forecast
  └─ reads ← arr_forecast_prophet (latest run_date)
  └─ returns JSON to frontend
```

### Option A — FastAPI reads from Delta table (recommended)

Update `backend/routes/arr.py` (or `main.py`) to query the pre-computed table.
No model runs in the app. Response time drops from ~30s (Prophet fit) to <1s (SQL read).

### Option B — Databricks Apps host the notebook directly

Wrap this notebook as a Databricks App using the Databricks Apps service.
The app surfaces a REST endpoint automatically; the FastAPI backend calls that endpoint.